# Agent Skills

## Northstar: reusable, governed operational playbooks

An incident agent needs a reusable procedure for collecting release evidence without loading every operational guide into context. We discover an eligible skill through metadata, progressively load its procedure, and show how it composes with MCP tools and bounded subagents.

**Outcomes:** distinguish tools from skills; design skill descriptions/discovery/libraries; apply progressive loading, composition, routing, governance, and security; and connect skills safely to MCP and subagents. All default cells are deterministic and credential-free.

![Agent Skills architecture](../../../assets/agent-skills-architecture.svg)

A skill is a versioned procedural package, not an authority grant. The host filters discovery with policy, activates a specific reviewed version, loads supporting material only when required, and separately authorizes every tool or subagent action.

## Step 1 — tools versus reusable procedural capabilities

A tool is a typed operation such as `read_deployment`. A skill is the reusable procedure that explains when to collect deployment evidence, what to validate, what artifact to produce, what failure to escalate, and which approved tools may be relevant. The Agent Skills format commonly packages this in a directory with `SKILL.md`, optional scripts, references, and assets.

Start with a narrow trigger and a specific outcome. A broad prompt called *incident expert* is not a dependable skill.

In [1]:
from pathlib import Path
import sys
TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'advanced' / '14-agent-skills'
sys.path.insert(0, str(TOPIC))
from lab import LIBRARY, activate, compose, discover

for skill in LIBRARY:
    print(skill.name, '→', skill.description)
assert all(skill.description for skill in LIBRARY)

incident-analysis → Use for evidence-backed incident triage and proposals.
customer-impact → Use for SLA-safe customer-impact assessment.


## Step 2 — discovery, routing, and dynamic loading

Discovery should first inspect lightweight metadata: name, description, owner/version/provenance, compatibility, risk, required data/tools, and trigger. A semantic route can rank candidates, but deterministic policy decides whether a skill is eligible for this tenant and scope. Only after a safe match does the host load `SKILL.md`; deeper references and scripts load just-in-time. This progressive disclosure reduces context cost and exposure, but does not make dynamically loaded content trusted.

In [2]:
permitted = {'read_metrics', 'read_deployment'}
matches = discover('incident triage with evidence', permitted)
print(matches)
active = activate(matches[0], permitted)
print(active)
assert active['load'] == ['SKILL.md', 'runbook.md']

try:
    activate(LIBRARY[1], permitted)
except PermissionError as error:
    print('policy blocked:', error)

[Skill(name='incident-analysis', description='Use for evidence-backed incident triage and proposals.', allowed_tools=frozenset({'read_metrics', 'read_deployment'}), references=('runbook.md',))]
{'skill': 'incident-analysis', 'load': ['SKILL.md', 'runbook.md'], 'tools': ['read_deployment', 'read_metrics']}
policy blocked: skill requests unavailable tool


## Step 3 — skill libraries and procedural knowledge

A skill library is a governed catalog: owner, source/provenance, version, trigger, tool/data requirements, risk, compatibility, tests/evaluations, dependencies, deprecation, and revocation. Skills hold procedural knowledge (how to perform an approved workflow). They are not semantic facts or an unrestricted memory store. Trace selection and activated version so an incident can be reproduced and a vulnerable skill can be revoked.

Treat instructions, scripts, references, and assets as supply-chain inputs. Review and scan them; sandbox scripts; constrain allowed tools; and never let a skill silently alter identity, permission, policy, spend, or approval requirements.

## Step 4 — composition, MCP, and subagents

Composition needs contracts: inputs, outputs, precedence, shared state, scope, budget, conflicts, and terminal behavior. Concatenating two instruction files is not composition. The conservative default is **not** to union privileges: composition intersects tool permissions, then the application may grant a precisely authorized extra capability.

MCP provides tool/context contracts; a skill provides the procedure for choosing approved MCP operations. A subagent can receive a skill/version plus a bounded task contract, but must have its own minimized tool/data scope, deadline, budget, and expected artifact.

In [3]:
# The two built-in skills have no shared tool: composition grants no implicit privilege.
combined = compose(list(LIBRARY))
print('conservative composed tools:', combined)
assert combined == frozenset()

contract = {
    'skill': 'incident-analysis@1.0.0', 'task': 'Assess deploy-842 contribution',
    'tools': ['read_metrics', 'read_deployment'], 'budget_calls': 2,
    'expected_artifact': 'cited release assessment', 'stop': 'publish or escalate',
}
contract

conservative composed tools: frozenset()


{'skill': 'incident-analysis@1.0.0',
 'task': 'Assess deploy-842 contribution',
 'tools': ['read_metrics', 'read_deployment'],
 'budget_calls': 2,
 'expected_artifact': 'cited release assessment',
 'stop': 'publish or escalate'}

## Step 5 — evaluate and operate skills

Test discovery precision/recall, unsafe selection rate, activation success, output/trajectory quality, tool policy violations, context/token cost, and regressions by skill version. A useful skill must improve repeatability or outcomes over a clear baseline, not merely add instructions. Keep a fallback when no eligible skill matches, and route ambiguity to a human or a bounded generalist.

**Exercises:** write a `SKILL.md` for customer-impact assessment; add a version and revocation check; simulate a malicious reference that asks to expand tool scope; compose two compatible skills with a typed handoff; and compare direct prompting against a skill on a fixed evaluation set.

References: [Agent Skills specification](https://github.com/agentskills/agentskills/blob/main/docs/specification.mdx), [Agent Skills project](https://github.com/agentskills/agentskills), [OpenAI Skills](https://openai.com/academy/skills/), [skills research survey](https://arxiv.org/abs/2602.12430), [MCP](https://modelcontextprotocol.io/specification/), [A2A](https://a2a-protocol.org/latest/).